# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ayushmansaha1013/Fly_rank_ML_internship_repo/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [5]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

con = duckdb.connect()

# Set up native Hugging Face secret
con.execute(f"""
CREATE SECRET hf_token (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
);
""")

# Target month=2026-03 inside the partitioned parquet table
DATA_PATH = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

# Query test
print(con.sql(f"SELECT COUNT(*) FROM read_parquet('{DATA_PATH}')").fetchdf())

   count_star()
0       9841378


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
q1 = f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COUNT(*) as row_count
FROM read_parquet('{DATA_PATH}')
GROUP BY client_hash_id, content_hash_id, report_date
HAVING COUNT(*) > 1
"""
duplicates = con.execute(q1).fetchdf()
print(f"Grain Duplicate Count: {len(duplicates)}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Grain Duplicate Count: 0


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
q2 = f"""
SELECT
    COUNT(*) as total_rows,
    MIN(report_date) as min_date,
    MAX(report_date) as max_date,
    COUNT(DISTINCT report_date) as total_days
FROM read_parquet('{DATA_PATH}')
"""
print(con.execute(q2).fetchdf())

   total_rows   min_date   max_date  total_days
0     9841378 2026-03-01 2026-03-31          31


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Check exact column names and data types in the table
print(con.execute(f"DESCRIBE SELECT * FROM read_parquet('{DATA_PATH}')").fetchdf()[['column_name', 'column_type']])


                 column_name column_type
0                report_date        DATE
1             client_hash_id     VARCHAR
2            content_hash_id     VARCHAR
3             client_has_gsc     BOOLEAN
4             client_has_ga4     BOOLEAN
5         gsc_data_available     BOOLEAN
6         ga4_data_available     BOOLEAN
7            gsc_impressions      BIGINT
8                 gsc_clicks      BIGINT
9           gsc_sum_position      BIGINT
10          gsc_avg_position      DOUBLE
11             ga4_pageviews      BIGINT
12              ga4_sessions      BIGINT
13                 ga4_users      BIGINT
14      ga4_engaged_sessions      BIGINT
15  ga4_total_engagement_sec      BIGINT
16          sessions_organic      BIGINT
17           sessions_direct      BIGINT
18         sessions_referral      BIGINT
19           sessions_social      BIGINT
20             sessions_paid      BIGINT
21               sessions_ai      BIGINT
22                ai_chatgpt      BIGINT
23             a

In [10]:
# Query 3: Show row survival using gsc_data_available IS TRUE (or both GSC & GA4)
q3 = f"""
SELECT
    COUNT(*) as total_rows,

    -- Rows where Search Console data is available
    COUNT(CASE WHEN gsc_data_available IS TRUE THEN 1 END) as gsc_available_rows,
    ROUND(COUNT(CASE WHEN gsc_data_available IS TRUE THEN 1 END) * 100.0 / COUNT(*), 2) as gsc_survival_pct,

    -- Rows where Analytics (GA4) data is available
    COUNT(CASE WHEN ga4_data_available IS TRUE THEN 1 END) as ga4_available_rows,
    ROUND(COUNT(CASE WHEN ga4_data_available IS TRUE THEN 1 END) * 100.0 / COUNT(*), 2) as ga4_survival_pct,

    -- Rows where BOTH sources are available
    COUNT(CASE WHEN gsc_data_available IS TRUE AND ga4_data_available IS TRUE THEN 1 END) as both_available_rows,
    ROUND(COUNT(CASE WHEN gsc_data_available IS TRUE AND ga4_data_available IS TRUE THEN 1 END) * 100.0 / COUNT(*), 2) as both_survival_pct
FROM read_parquet('{DATA_PATH}')
"""

df_q3 = con.execute(q3).fetchdf()
print(df_q3)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  gsc_available_rows  gsc_survival_pct  ga4_available_rows  \
0     9841378             3611061             36.69              413966   

   ga4_survival_pct  both_available_rows  both_survival_pct  
0              4.21               364347                3.7  


In [11]:
# Build 5-Feature Frame using actual schema column names
feature_query = f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,

    -- Label: Target variable (1 if GSC clicks > 0, else 0)
    CASE WHEN gsc_clicks > 0 THEN 1 ELSE 0 END as label_clicked,

    -- Feature 1: Historical average CTR prior to current date
    AVG(gsc_clicks * 1.0 / NULLIF(gsc_impressions, 0)) OVER (
        PARTITION BY content_hash_id ORDER BY report_date
        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    ) as f1_hist_ctr,

    -- Feature 2: Historical average position prior to current date
    AVG(gsc_avg_position) OVER (
        PARTITION BY content_hash_id ORDER BY report_date
        ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
    ) as f2_7d_avg_position,

    -- Feature 3: Hash key string length (proxy feature)
    LENGTH(content_hash_id) as f3_hash_length,

    -- Feature 4: Day of the week from report date
    DAYOFWEEK(report_date) as f4_day_of_week,

    -- Feature 5: Rolling 7-day total impressions prior to current date
    SUM(gsc_impressions) OVER (
        PARTITION BY content_hash_id ORDER BY report_date
        ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
    ) as f5_7d_sum_impressions
FROM read_parquet('{DATA_PATH}')
WHERE gsc_data_available IS TRUE
"""

df_features = con.execute(feature_query).fetchdf()

# Machine Learning & The Leakage Trap Experiment
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

df_model = df_features.fillna(0)

X_honest = df_model[['f1_hist_ctr', 'f2_7d_avg_position', 'f3_hash_length', 'f4_day_of_week', 'f5_7d_sum_impressions']]
y = df_model['label_clicked']

# 1. Honest Baseline Model
clf = lgb.LGBMClassifier(random_state=42, verbose=-1)
clf.fit(X_honest, y)
honest_auc = roc_auc_score(y, clf.predict_proba(X_honest)[:, 1])
print(f"Honest Model ROC-AUC Score: {honest_auc:.4f}")

# 2. Add ONE Label-Derived Leaked Feature (THE TRAP)
df_model['trap_leaked_feature'] = df_model['label_clicked'] * 0.99 + 0.01

X_leaked = df_model[['f1_hist_ctr', 'f2_7d_avg_position', 'f3_hash_length', 'f4_day_of_week', 'f5_7d_sum_impressions', 'trap_leaked_feature']]
clf.fit(X_leaked, y)
leaked_auc = roc_auc_score(y, clf.predict_proba(X_leaked)[:, 1])
print(f"Leaked (Trap) Model ROC-AUC Score: {leaked_auc:.4f}  <-- Artificial Jump!")

# 3. Delete the Leaked Feature to Keep Model Clean
df_model.drop(columns=['trap_leaked_feature'], inplace=True)
print("Deleted leaked column successfully. Retained honest baseline.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Honest Model ROC-AUC Score: 0.8798
Leaked (Trap) Model ROC-AUC Score: 1.0000  <-- Artificial Jump!
Deleted leaked column successfully. Retained honest baseline.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.